In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

In [9]:
# Load the trained model
model = load_model('churn_model.h5')

# Load the scaler
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)
# Load the One-Hot Encoder
with open('onehot_encoder.pkl', 'rb') as file:
    ohe = pickle.load(file)

# load the label encoder
with open('label_encoder.pkl', 'rb') as file:
    le = pickle.load(file)

In [27]:
### Example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [28]:
# Apply one-hot encoding to categorical features
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [29]:
geography_encoded = ohe.transform(input_df[['Geography']])
geography_encoded = geography_encoded.toarray()
ohe.get_feature_names_out()

geography_encoded_df = pd.DataFrame(geography_encoded, columns=ohe.get_feature_names_out(['Geography']))
input_df = pd.concat([ input_df.drop('Geography', axis=1), geography_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [30]:
# Encode Gender
input_df['Gender'] = le.transform(input_df['Gender'])
input_df = pd.concat([input_df.reset_index(drop=True)], axis=1)


In [31]:
### Scale the input features
scaled_input = scaler.transform(input_df)
scaled_input

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [32]:
### Predict churn probability

prediction = model.predict(scaled_input)

print(f"Churn Probability: {prediction[0][0]:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step
Churn Probability: 0.0551
